In [1]:
import numpy as np
import pandas as pd
import re
import pickle

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

In [7]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [8]:
df = pd.read_csv("BBC_Text_CLS.csv")
df.head()

,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


In [9]:
print(df.shape)
print(df['labels'].value_counts())

(2225, 2)
labels
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


In [10]:
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove special chars
    tokens = word_tokenize(text)  # tokenize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # lemmatize
    return " ".join(tokens)

In [11]:
df['clean_text'] = df['text'].apply(clean_text)
df.head()

,text,labels,clean_text
0,Ad sales boost Time Warner profit\n\nQuarterly...,business,ad sale boost time warner profit quarterly pro...
1,Dollar gains on Greenspan speech\n\nThe dollar...,business,dollar gain on greenspan speech the dollar ha ...
2,Yukos unit buyer faces loan claim\n\nThe owner...,business,yukos unit buyer face loan claim the owner of ...
3,High fuel prices hit BA's profits\n\nBritish A...,business,high fuel price hit ba s profit british airway...
4,Pernod takeover talk lifts Domecq\n\nShares in...,business,pernod takeover talk lift domecq share in uk d...


In [15]:
X = df['clean_text']
y = df['labels']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
X

,clean_text
0,ad sale boost time warner profit quarterly pro...
1,dollar gain on greenspan speech the dollar ha ...
2,yukos unit buyer face loan claim the owner of ...
3,high fuel price hit ba s profit british airway...
4,pernod takeover talk lift domecq share in uk d...
...,...
2220,bt program to beat dialler scam bt is introduc...
2221,spam e mail tempt net shopper computer user ac...
2222,be careful how you code a new european directi...
2223,u cyber security chief resigns the man making ...


In [17]:
vectorizer = CountVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [18]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

In [19]:
# Accuracy
train_acc = model.score(X_train_vec, y_train)
test_acc = model.score(X_test_vec, y_test)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

y_pred = model.predict(X_test_vec)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Train Accuracy: 0.9921348314606742
Test Accuracy: 0.9707865168539326

Classification Report:

               precision    recall  f1-score   support

     business       0.97      0.95      0.96       115
entertainment       0.99      0.96      0.97        72
     politics       0.94      0.97      0.95        76
        sport       1.00      0.99      1.00       102
         tech       0.95      0.99      0.97        80

     accuracy                           0.97       445
    macro avg       0.97      0.97      0.97       445
 weighted avg       0.97      0.97      0.97       445



In [20]:
def predict(text):
    text = clean_text(text)
    text_vec = vectorizer.transform([text])
    return model.predict(text_vec)[0]

sample = "The government is planning new economic reforms"
print("Prediction:", predict(sample))

Prediction: business


In [22]:
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))